In [ ]:
import os
import glob
import json

import numpy as np


In [ ]:
ROOT_PTH = "/home/klymy/data/cam4_revB"

HOLDOUT_PECENT = 0.1
VAL_PECENT = 0.1

In [ ]:
def is_in_subset(pth: str, holdout_names: list[str]) -> bool:
    for i in holdout_names:
        if f"{i}_" in pth:
            return True
    return False

In [ ]:
def get_data(name: str) -> str:
    name = name.replace("4.2_H_cam_", "").replace(".tif", "")
    parts = name.split("_")
    n = 2
    if not parts[n].isnumeric():
        n += 1
    return "_".join(parts[:n])

In [ ]:
img_pths = glob.glob(os.path.join(ROOT_PTH, "*/*.tif"), recursive=True)
img_names = [os.path.basename(pth) for pth in img_pths]
N = len(img_names)
N

In [ ]:
height_location_names = [i.replace("4.2_H_cam_", "").replace(".tif", "") for i in img_names]

In [ ]:
height_location_names = [get_data(i) for i in img_names]

In [ ]:
unique_names, counts = np.unique(height_location_names, return_counts=True, )

In [ ]:
sorted_idxs = np.argsort(counts)
unique_names = unique_names[sorted_idxs]
counts = counts[sorted_idxs]

In [ ]:
holdout_subset = np.cumsum(counts) < N * HOLDOUT_PECENT
holdout_names = unique_names[holdout_subset]

holdout_pths = [i for i in img_pths if is_in_subset(i, holdout_names)]
working_pths = [i for i in img_pths if i not in holdout_pths]
len(holdout_pths), len(working_pths)

In [ ]:
np.random.seed(42)
np.random.shuffle(working_pths)
val_pths = working_pths[:int(len(working_pths) * VAL_PECENT)]
training_pths = working_pths[int(len(working_pths) * VAL_PECENT):]
len(training_pths), len(val_pths), len(holdout_pths), len(training_pths) + len(val_pths) + len(holdout_pths)

In [ ]:
with open(os.path.join(ROOT_PTH, "train_val_holdout_subsets_batches_1_3.json"), "w") as f:
    json.dump({
        "train": training_pths,
        "val": val_pths,
        "holdout": holdout_pths,
    }, f)